# Phase 4: per-site circuit resolution

1. Restrict to `CLEAN_SITE_IDS` (1 or 3 `ac_load_net` circuits) -- computed fresh here, the same way notebook 3 section 3d did.
   `OTHER_COUNT_SITE_IDS` is DEFERRED to a later phase, not built here.
2. For each site, resolve its `ac_load_net`/`pv_site_net` circuits: drop
   confirmed duplicates (`ami_taxonomy.find_duplicate_circuits`) and
   inactive circuits (`ami_taxonomy.find_inactive_circuits`), and flag any
   surviving circuit for the device/meter-model power correction
   (`ami_resample.energy_granularity_and_implied_interval`). All of this is
   `ami_resolution.resolve_site_circuits` -- this notebook orchestrates and
   narrates only; see that module's docstring for the exact rules (in
   particular: a CROSS-type duplicate always drops the load-side member; a
   SAME-type duplicate group keeps the most-covered circuit and flags the
   SITE for manual review, since Phase 3 never established a rule for that
   case).
3. Build the interval-level table (`ami_resolution.build_interval_table`)
   and the site-level metadata sidecar with its resolution audit trail
   (`ami_resolution.build_site_metadata`).
4. Run a coverage report (`ami_resolution.build_coverage_report`) -- how
   many sites needed no intervention vs. were auto-resolved vs. were
   flagged for manual review.

**Scope of this first run (agreed 2026-08-27):** a small validation batch --
`N_VALIDATION_SITES` sites sampled from `CLEAN_SITE_IDS`, one day -- the
same size and day as Phase 3 Section 8b's fleet scan, so results are
directly comparable to the 21.5%/2% duplicate/inactive numbers already
found. Not the full 14,353-site fleet; scale up only after reviewing this
batch's audit trail and both tables.

**Reactive power** has no raw instantaneous column in `ts` at all (only
`energy_reactive`) -- `build_interval_table` derives it the same way for
every kept circuit: `energy_reactive * 60 / interval_minutes`, using each
circuit's own implied interval where flagged, the nominal interval
otherwise.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as Config
from bms_sa_review.ami_data_analysis.lib import ami_athena as Athena
from bms_sa_review.ami_data_analysis.lib import ami_taxonomy as Taxonomy
from bms_sa_review.ami_data_analysis.lib import ami_resample as Resample
from bms_sa_review.ami_data_analysis.lib import ami_resolution as Resolution
from bms_sa_review.ami_data_analysis.lib import ami_signal as Signal
from bms_sa_review.ami_data_analysis.lib import ami_plots as Plots

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

Athena.reset_scan_log()
Athena.require_credentials()
print("Credentials OK. Starting Phase 4 site resolution.")


## 1. Cohort: recompute `CLEAN_SITE_IDS` fresh

Mirrors notebook 3 sections 3a/3d exactly (same queries, same restriction to
sites with at least one PV-side AND one load-side circuit_type present),
kept self-contained here rather than depending on notebook 3 having been run
in the same kernel session.


In [ ]:
raw_circuit_type_counts = Athena.aq(
    """
    SELECT circuit_type, is_pv, count(*) AS n_circuits, count(DISTINCT site_id) AS n_sites
    FROM meta_up23c
    GROUP BY circuit_type, is_pv
    """,
    database=Config.SAI, label="circuit_type census (Phase 4)",
)
census = Taxonomy.summarise_circuit_types(raw_circuit_type_counts)
census = Taxonomy.flag_suspected_aggregates(census)
suspects = census[census.suspected_aggregate]

load_candidate_type = "ac_load_net"
pv_candidate_type = "pv_site_net"
assert load_candidate_type in suspects.circuit_type.tolist(), (
    f"{load_candidate_type!r} not flagged as a suspected aggregate this run -- "
    "the fleet's circuit_type census may have changed; re-check before proceeding."
)
assert pv_candidate_type in suspects.circuit_type.tolist(), (
    f"{pv_candidate_type!r} not flagged as a suspected aggregate this run -- "
    "the fleet's circuit_type census may have changed; re-check before proceeding."
)
print(f"Load-side candidate: {load_candidate_type!r}. PV-side candidate: {pv_candidate_type!r}.")


In [ ]:
meta_local = Athena.aq(
    """
    SELECT circuit_id, site_id, circuit_type, is_pv, circuit_polarity, ac_capacity_kw,
           dc_capacity_kw, export_limit_kw, inverter_count,
           device_id, m_id, device_type, voltage_class, min_time, max_time, s_99,
           state, postcode, dnsp_name, flex_export_detected, manufacturer, model,
           monitoring_start, pv_install_date
    FROM meta_up23c
    """,
    database=Config.SAI, label="meta_up23c full projection (Phase 4)",
)
print(f"meta_local: {len(meta_local):,} rows, {meta_local.circuit_id.nunique():,} distinct circuits")

cohort = Taxonomy.cohort_completeness(meta_local)
pv_types = census[census.is_pv.astype(bool)].circuit_type.tolist()
load_types = census[~census.is_pv.astype(bool)].circuit_type.tolist()

pv_cols = [t for t in pv_types if t in cohort.columns]
load_cols = [t for t in load_types if t in cohort.columns]
pv_present = cohort[pv_cols].sum(axis=1) > 0 if pv_cols else pd.Series(False, index=cohort.index)
load_present = cohort[load_cols].sum(axis=1) > 0 if load_cols else pd.Series(False, index=cohort.index)
cohort_with_both = cohort[pv_present & load_present].reset_index(drop=True)
print(f"{len(cohort_with_both):,} of {len(cohort):,} sites have at least one PV-side AND "
      "one load-side circuit_type present.")

pv_cohort = cohort[pv_present].reset_index(drop=True)


In [ ]:
CLEAN_AC_LOAD_NET_COUNTS = (1, 3)

counts_by_site = pv_cohort[["site_id", load_candidate_type]].rename(
    columns={load_candidate_type: "n_circuits"}
)
counts_by_site = counts_by_site[counts_by_site.n_circuits > 0]

CLEAN_SITE_IDS, OTHER_COUNT_SITE_IDS = Resolution.classify_circuit_counts(
    counts_by_site, clean_counts=CLEAN_AC_LOAD_NET_COUNTS
)
print(f"{len(CLEAN_SITE_IDS):,} CLEAN_SITE_IDS (1 or 3 `{load_candidate_type}` circuits).")
print(f"{len(OTHER_COUNT_SITE_IDS):,} OTHER_COUNT_SITE_IDS -- DEFERRED to a later phase, "
      "not built here (agreed 2026-08-27).")


## 2. Pull a validation batch: `N_VALIDATION_SITES` sites, one day

Same sampling approach and same day as Phase 3 Section 8b's fleet scan, so
this run's coverage numbers are directly comparable to the 21.5%/2%
duplicate/inactive findings already established. Two batched queries (one
per `is_pv` side), not one per site -- `find_duplicate_circuits`/
`find_inactive_circuits`/`energy_granularity_and_implied_interval` all group
internally by site/circuit already.


In [ ]:
N_VALIDATION_SITES = 200  # small validation batch -- see section intro
VALIDATION_YEAR, VALIDATION_MONTH = 2025, 6
VALIDATION_DAY_START = "2025-06-01 00:00:00"
VALIDATION_DAY_END = "2025-06-02 00:00:00"

site_pool = pd.Series(CLEAN_SITE_IDS)
validation_site_ids = sorted(
    site_pool.sample(n=min(N_VALIDATION_SITES, len(site_pool)), random_state=0).tolist()
)
print(f"Validation batch: {len(validation_site_ids):,} of {len(CLEAN_SITE_IDS):,} "
      f"CLEAN_SITE_IDS, {VALIDATION_YEAR}-{VALIDATION_MONTH:02d} (one day).")

KEPT_TYPES = [load_candidate_type, pv_candidate_type]
batch_meta = meta_local[
    meta_local.site_id.isin(validation_site_ids) & meta_local.circuit_type.isin(KEPT_TYPES)
]

batch_pulls = []
for is_pv_value in (False, True):
    side_ids = batch_meta[batch_meta.is_pv == is_pv_value].circuit_id.tolist()
    if not side_ids:
        continue
    id_list_sql = ", ".join(str(int(c)) for c in side_ids)
    is_pv_sql = "true" if is_pv_value else "false"
    pulled = Athena.aq(
        f"""
        SELECT circuit_id, t_stamp, power, energy, energy_reactive, energy_import,
               energy_export, energy_reactive_import, energy_reactive_export,
               power_factor, voltage, current
        FROM ts
        WHERE year = {VALIDATION_YEAR} AND month = {VALIDATION_MONTH} AND is_pv = {is_pv_sql}
          AND circuit_id IN ({id_list_sql})
          AND t_stamp >= TIMESTAMP '{VALIDATION_DAY_START}'
          AND t_stamp <  TIMESTAMP '{VALIDATION_DAY_END}'
        ORDER BY circuit_id, t_stamp
        """,
        database=Config.SAI,
        label=f"Phase 4 validation batch, is_pv={is_pv_value}",
    )
    batch_pulls.append(pulled)

batch_sample = pd.concat(batch_pulls, ignore_index=True) if batch_pulls else pd.DataFrame()
if len(batch_sample):
    # find_duplicate_circuits/find_inactive_circuits need site_id (and the
    # resolution/interval-table steps need circuit_type/device_id too) --
    # the `ts` pull itself never carries these, only `circuit_id` (mirrors
    # notebook 3 section 8b's identical merge-after-pull pattern).
    batch_sample = batch_sample.merge(
        batch_meta[["circuit_id", "site_id", "circuit_type", "device_id", "circuit_polarity"]],
        on="circuit_id", how="left",
    )
print(f"{len(batch_sample):,} rows, {batch_sample.circuit_id.nunique() if len(batch_sample) else 0:,} "
      "circuits pulled.")


## 3. Resolve every site's circuits

One call across the whole batch (not a per-site loop) -- see
`ami_resolution.resolve_site_circuits`'s docstring for the exact
duplicate/inactive/correction rules.


In [ ]:
resolution = Resolution.resolve_site_circuits(
    batch_meta, batch_sample,
    load_type=load_candidate_type, pv_type=pv_candidate_type,
    nominal_interval_minutes=Config.SOURCE_INTERVAL_MINUTES,
)
display(resolution.head(20))

n_dropped = int((~resolution.kept).sum())
print(f"\n{len(resolution):,} candidate circuit(s) across {resolution.site_id.nunique():,} sites -- "
      f"{n_dropped:,} dropped.")
if n_dropped:
    display(resolution.loc[~resolution.kept, ["site_id", "circuit_id", "circuit_type", "drop_reason"]])


## 4. Build the two output tables


In [ ]:
interval_table = Resolution.build_interval_table(
    batch_sample, resolution, nominal_interval_minutes=Config.SOURCE_INTERVAL_MINUTES,
)
print(f"interval_table: {len(interval_table):,} rows, "
      f"{interval_table.circuit_id.nunique():,} circuits, "
      f"{interval_table.site_id.nunique():,} sites.")
display(interval_table.head(10))


In [ ]:
SITE_METADATA_COLUMNS = [
    "site_id", "state", "postcode", "dnsp_name", "ac_capacity_kw", "dc_capacity_kw",
    "export_limit_kw", "inverter_count", "voltage_class", "manufacturer", "model",
    "flex_export_detected", "monitoring_start", "pv_install_date",
]
site_level_meta = (
    meta_local[meta_local.site_id.isin(validation_site_ids)]
    [[c for c in SITE_METADATA_COLUMNS if c in meta_local.columns]]
    .drop_duplicates(subset="site_id")
)

site_metadata = Resolution.build_site_metadata(site_level_meta, resolution)
print(f"site_metadata: {len(site_metadata):,} rows (of {len(validation_site_ids):,} "
      "sites in the validation batch).")
display(site_metadata.head(10))


## 5. Coverage report


In [ ]:
coverage_report = Resolution.build_coverage_report(resolution)
display(pd.DataFrame([coverage_report]))

print(
    f"\n{coverage_report['n_no_intervention']:,} of {coverage_report['n_sites']:,} sites needed no "
    "intervention.\n"
    f"{coverage_report['n_auto_resolved_duplicate_cross_type']:,} auto-resolved via a cross-type "
    "duplicate drop (deterministic rule -- see module docstring).\n"
    f"{coverage_report['n_auto_resolved_inactive']:,} auto-resolved via an inactive-circuit drop.\n"
    f"{coverage_report['n_flagged_manual_review']:,} flagged for manual review (a same-type "
    "duplicate group -- no established rule for which circuit is 'real').\n"
    f"{coverage_report['n_power_correction_applied']:,} kept circuit(s) got the device/meter-model "
    "power correction applied."
)


## 7. Load reconstruction & storage sanity check (new, 2026-08-27)

`ac_load_net` is not gross house consumption -- it is already net of PV
(negative when the site is exporting more solar than it's drawing). To
recover the true, PV-independent house load, add the sign-corrected PV
generation back: `house_load = ac_load_net (signed) + pv_site_net (signed)`.

This has only been confirmed BY EYE, on one real site -- not proven at
scale the way Phase 3 proved everything else (arithmetically, across many
sites). This section runs that proof, reusing the SAME validation batch
already pulled above (no new Athena query):

1. `Signal.reconstruct_gross_load` applies `circuit_polarity`, sums
   `ac_load_net` across any phases at a site, sums `pv_site_net`
   analogously, and adds them -- one candidate `reconstructed_load` per
   (site_id, t_stamp).
2. `Signal.evaluate_load_reconstruction` checks whether that candidate is
   PLAUSIBLE: true house load should not go meaningfully negative, and a
   NIGHT-time violation (PV is ~0 overnight, Section 9) is the sharper
   signal -- it can't be explained away by a near-simultaneous PV reading
   in the same sum, so it means either the sign convention is wrong at that
   site, or an undocumented battery/EV effect is netted into `ac_load_net`
   behind the same CT (no separate circuit_id to catch by name).
3. `Signal.sites_with_storage_circuits` catches the OTHER half: an
   EXPLICIT, separately-metered battery/EV circuit_type, detectable by name
   alone. A site can fail either check independently, or both.

Neither check is applied to the two Phase 4 output tables -- this is a
diagnostic only, informing (not yet resolving) `SIGN_CONVENTION_RESOLVED`
and `STORAGE_HANDLING` in `ami_config.py`, and scoping whichever future
phase actually builds `gross_load`/`pv_generation` as named signals.


In [ ]:
storage_site_ids = Signal.sites_with_storage_circuits(meta_local)
print(f"{len(storage_site_ids):,} site(s) in the FULL meta_up23c pull have an explicit, "
      "name-detected battery/EV circuit_type (not limited to the validation batch).")

batch_storage_site_ids = sorted(set(storage_site_ids) & set(validation_site_ids))
print(f"{len(batch_storage_site_ids):,} of those are in this validation batch: "
      f"{batch_storage_site_ids}")


In [ ]:
circuit_polarity_lookup = meta_local[["circuit_id", "circuit_polarity"]].drop_duplicates("circuit_id")

reconstructed = Signal.reconstruct_gross_load(interval_table, circuit_polarity_lookup)
reconstructed["t_stamp"] = Plots.to_aest(reconstructed["t_stamp"])

reconstruction_report = Signal.evaluate_load_reconstruction(reconstructed)
display(reconstruction_report)

n_flagged = int(reconstruction_report.likely_storage_or_sign_issue.sum())
print(f"\n{n_flagged:,} of {len(reconstruction_report):,} sites show a NIGHT-time negative "
      "reconstructed load -- a sign-convention problem or an undocumented storage effect at "
      "that site, not explainable by PV (which is ~0 overnight).")

flagged_site_ids = set(reconstruction_report.loc[reconstruction_report.likely_storage_or_sign_issue, "site_id"])
overlap_with_named_storage = flagged_site_ids & set(batch_storage_site_ids)
only_reconstruction_flagged = flagged_site_ids - set(batch_storage_site_ids)
print(f"{len(overlap_with_named_storage):,} of those also have an explicit storage circuit_type "
      "(both checks agree).")
print(f"{len(only_reconstruction_flagged):,} are flagged ONLY by the reconstruction check -- "
      "candidates for an invisible, same-CT battery effect: "
      f"{sorted(only_reconstruction_flagged)}")


**Reading this report:** a site with `likely_storage_or_sign_issue=False` and a low
`share_negative_all` is consistent with the delta hypothesis holding cleanly there.
A site flagged here (by either check) should be excluded from the "clean" core
ground-truth cohort until it's understood, not silently included with a
contaminated `house_load` -- see the site-metadata audit trail for where a
future phase should record this exclusion.


### 7b. Drop flagged sites, rebuild the two output tables (new, 2026-08-27)

Simplest and most conservative response to a flagged site: drop it
entirely. Whether the actual cause is a per-circuit sign/polarity bug (site
898760152 in the real run: one circuit permanently mirrored, day and
night) or a genuine behind-the-meter battery (site 1267630483: a circuit
cycling both signs overnight with no PV to explain it), `ac_load_net` is
not trustworthy as pure house load at that site either way -- distinguishing
the two only matters for salvaging a site later, not for justifying the
drop now. `ami_resolution.exclude_flagged_sites` drops every surviving
circuit at each flagged site, while preserving whatever more specific
`drop_reason` a circuit already had (a site can be BOTH a duplicate-heavy
site AND flagged for storage -- the audit trail keeps both facts visible).


In [ ]:
excluded_site_ids = set(batch_storage_site_ids) | set(flagged_site_ids)
print(f"Excluding {len(excluded_site_ids):,} site(s) entirely: {sorted(excluded_site_ids)}")

final_resolution = Resolution.exclude_flagged_sites(resolution, excluded_site_ids)

# Rebuild both output tables and the coverage report from the FINAL,
# storage/sign-issue-excluded resolution -- these are what actually feed
# the synthetic AMI build, not the pre-exclusion versions above.
interval_table = Resolution.build_interval_table(
    batch_sample, final_resolution, nominal_interval_minutes=Config.SOURCE_INTERVAL_MINUTES,
)
site_metadata = Resolution.build_site_metadata(site_level_meta, final_resolution)
coverage_report = Resolution.build_coverage_report(final_resolution)

display(pd.DataFrame([coverage_report]))
print(f"\nFinal interval_table: {len(interval_table):,} rows, "
      f"{interval_table.site_id.nunique():,} sites (of {len(validation_site_ids):,} "
      "in the validation batch).")


## 6. Save small, human-reviewable samples to `artefacts/`

Full tables at this validation-batch scale are small enough to keep whole;
a full-fleet run would need a different (Parquet/DuckDB) storage decision,
out of scope for this pass.


In [ ]:
ARTEFACT_DIR = Config.ARTEFACT_DIR
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

interval_table.to_csv(ARTEFACT_DIR / "phase4_interval_table_validation_batch.csv", index=False)
site_metadata.to_csv(ARTEFACT_DIR / "phase4_site_metadata_validation_batch.csv", index=False)
pd.DataFrame([coverage_report]).to_csv(ARTEFACT_DIR / "phase4_coverage_report_validation_batch.csv", index=False)
print(f"Saved interval table, site metadata, and coverage report to {ARTEFACT_DIR}")


In [ ]:
interval_table

In [ ]:
import matplotlib.pyplot as plt

plot_data = interval_table.loc[
    interval_table["site_id"].eq(898760152),
    ["t_stamp", "power", "circuit_id", "circuit_type"],
].copy()

plot_data["t_stamp"] = pd.to_datetime(plot_data["t_stamp"])
plot_data = plot_data.sort_values(["circuit_id", "t_stamp"])

fig, ax = plt.subplots(figsize=(14, 5))

for (circuit_id, circuit_type), circuit_data in plot_data.groupby(
    ["circuit_id", "circuit_type"],
    sort=True,
):
    ax.plot(
        circuit_data["t_stamp"],
        circuit_data["power"],
        label=f"{circuit_id} ({circuit_type})",
        linewidth=1.2,
    )

ax.set(
    title="Power by circuit for site ###",
    xlabel="Timestamp",
    ylabel="Power",
)

ax.legend(title="Circuit ID (Circuit Type)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
for site_id in (898760152, 1267630483):
    site_frame = batch_sample[batch_sample.site_id == site_id].copy()
    site_frame["power_signed"] = site_frame.power * site_frame.circuit_polarity
    Plots.plot_circuit_day(site_frame, title=f"site {site_id}")